# Feature Engineering — Field Asset Health Monitor Project

Stage 4 of the pipeline. Consumes `sensor_readings.parquet` + `failure_windows.csv`
(D10 contract) and produces the **model-ready feature table** — one row per time
window, columns = engineered health features — plus its saved artifact.

**Specification (from stage 3's feed-forward):**
- Features in BOTH directions: rising duty/oil (gradual leaks, F4-type) AND
  idle/low-duty indicators (step failures, F1-type).
- Multi-scale look-backs (hours → days; F3's faint signal, if any, is early).
- State-occupancy features (fraction of time per Motor_current mode), not just
  central tendency (6.4).
- Dynamics/variability features — F3's only remaining chance (OQ5).
- Instrument-health features as a separate axis (antiphase, rolling variance).
- Gap-aware: no window may bridge a recording gap.
- Train/eval design per D19: features must carry the label so healthy-only
  training and two-target evaluation are possible downstream.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from fahm import preprocessing as pp
from fahm import plotting as pl
from fahm import analysis as an
from fahm import features as ft          # NEW package module (see below)

cfg = pp.load_config("../configs/config.yaml")
df = pp.load_processed(cfg)
fw = pd.read_csv(cfg["paths"]["failure_windows"],
                 parse_dates=["start", "end", "maintenance"])
gaps = pp.find_gaps(df, cfg)
labels = an.label_windows(df, fw,
    degraded_periods=[("2020-04-18", "2020-04-30")],
    invalid_periods=[("2020-04-20 04:45", "2020-04-21 01:30")])
print(df.shape, "| labels:", dict(labels.value_counts()))

## 1. Design: the feature grid

Decisions to make and log BEFORE building (D20):
- **Window size** for the feature rows (candidate: 1h — fine enough for
  48h-precursor dynamics, coarse enough that 175 days ≈ 4,200 rows).
- **Gap rule:** windows are computed within contiguous segments only; a
  window crossing a gap > threshold is dropped (stage-1 gap inventory).
- **Look-back scales** for rolling features (candidate: 6h / 24h / 168h).
- **Window label** = majority label of its samples (invalid wins on any
  overlap — trust rule).

In [ ]:
# grid = ft.build_window_grid(df, cfg)        # window_start | window_end | segment_id
# grid_labels = ft.label_grid(grid, labels)     # majority label per window
# grid_labels.value_counts()

## 2. Feature families

Each family = one function in features.py returning columns for the grid.
Build one at a time; effect-size check after each (the stage-3 machinery
is the unit test for features).

| family | features (examples) | catches |
|---|---|---|
| **duty & state occupancy** | duty; frac time in motor modes (off/offloaded/loaded) | F4 up, F1 down, 6.4's mode story |
| **pressure dynamics** | TP3 idle-decay slope; cycle build rate; cycles/hour | the leak physics directly |
| **thermal** | oil median, oil trend (Δ over look-back), oil-per-duty | F4 ramp, F1 cool-idle |
| **variability (F3's chance)** | rolling std of duty & oil; cycle-duration variance; toggle rates | erratic-before-failing (OQ5) |
| **instrument health** | antiphase share; analog rolling-variance == 0 flags | Apr 20-type faults; trust mask |

In [ ]:
# feats = ft.build_features(df, grid, cfg)     # assembles all families
# feats.shape, feats.columns.tolist()

## 3. Feature validation — do engineered features separate better than raw?

The stage-3 effect-size machinery, now pointed at the features:
per-failure effect sizes of each feature family. Success criterion:
- F4/F1 signals at least as strong as raw sensors gave;
- **any F3 signal at all** in the variability family would be new information;
- instrument-health features fire on the Apr 20 window and nowhere unexpected.

In [ ]:
# an.prefail_effect_by_failure(feats_df, fw, feature_cols, grid_labels, hours=48)
# ... and 168h

## 4. Save the feature artifact

`features.parquet`: one row per window — features + label + window bounds.
The single input for stage 5 (modeling). Path in config (D11 convention).

In [ ]:
# path = ft.save_features(feats, cfg)

## Findings feed-forward (to stage 5 — modeling)
- feature table: <n windows x m features>, artifact at <path>
- which families separate, per failure: <...>
- F3 verdict after dynamics features: <detectable / confirmed sudden>
- instrument-health mask coverage: <...>
- training set definition: windows labeled healthy AND instrument-clean